# Integrales-03 — La bulle et la première divergence

Le carnet 01 calculait le triangle à une boucle : en $d = 4$, son comptage de
puissances vaut $4 - 2 \times 3 = -2$, et tout y restait fini. Le carnet 02
poussait au sunrise et voyait les polylogarithmes sortir — même comptage,
encore $-2$, cette fois en $d = 2$. Celui-ci descend d'un étage — **deux
propagateurs, la bulle** — et le comptage passe à $4 - 2 \times 2 = 0$, la
première fois que la série n'est plus protégée :

$$B(\Lambda) = \int_{|k| < \Lambda} \frac{d^4 k}{k^2 (k-p)^2}, \qquad p^2 = 1 .$$

C'est le plus petit graphe dont l'intégrale **diverge** : l'intégrande décroît
comme $1/k^4$ en $|k| \to \infty$, exactement la dimension du volume —
l'intégrale croît comme $\ln \Lambda^2$. Le programme du carnet :

1. **la mesure de la sphère $S^3$** — vérifiée numériquement avant tout usage
   (une première écriture de ce carnet utilisait $2\pi\sin^2\psi\, d\psi$ et
   rendait la moitié de l'intégrale : le contrôle importe) ;
2. **une forme fermée** par paramètre de Feynman, avec son domaine de validité
   exhibé (la boule dure ne commute pas au décalage — l'écart est mesuré, pas
   masqué) ;
3. **la divergence mesurée** : coefficient de $\ln \Lambda^2$ et constante
   finie, chacune ancrée dans une *planted truth* indépendante ;
4. **la régularisation soustractive** : un paramètre $\mu$, une limite finie.

Comme dans les carnets 01 et 02 : outil réel (`mpmath`, 30 chiffres), chaque
affirmation quantitative vit dans une sortie commise, les refus sont documentés.


## 1. La mesure de $S^3$

En quatre dimensions, $d^4 k = k^3\,dk\,d\Omega_3$ et la sphère unité se
paramètre par l'angle $\psi$ entre $k$ et $p$ :

$$d\Omega_3 = \sin^2\psi\, d\psi\, d\Omega_2, \qquad \int d\Omega_2 = 4\pi
\quad\Longrightarrow\quad \int_{S^3} d\Omega_3 = 4\pi \int_0^\pi \sin^2\psi\, d\psi = 2\pi^2 .$$

La cellule suivante vérifie l'aire par quadrature — c'est la *planted truth* de
la mesure : toute la suite en dépend.

In [1]:
from mpmath import mp, mpf, quad, pi, log, sqrt, sin, cos
mp.dps = 30
p2 = mpf(1)  # p^2 = 1 : l'echelle

# aire de S^3 par quadrature (planted truth de la mesure)
aire = quad(lambda psi: 4*pi*sin(psi)**2, [0, pi])
print("aire S^3 mesuree =", aire)
print("2 pi^2           =", 2*pi**2)
print("ecart relatif    =", abs((aire - 2*pi**2) / (2*pi**2)))

aire S^3 mesuree = 19.7392088021787172376689819998
2 pi^2           = 19.7392088021787172376689819998
ecart relatif    = 0.0


## 2. La forme fermée : paramètre de Feynman et complétion du carré

L'identité $\frac{1}{AB} = \int_0^1 \frac{dx}{(xA + (1-x)B)^2}$ avec
$A = k^2$, $B = (k-p)^2$ donne, après complétion du carré,

$$\frac{1}{k^2 (k-p)^2} = \int_0^1 \frac{dx}{\big((k - (1-x)p)^2 + \Delta\big)^2},
\qquad \Delta = x(1-x)\, p^2 .$$

**Avertissement honnête** : la forme fermée ci-dessous échange
$\int_{|k|<\Lambda}$ contre $\int_{|q|<\Lambda}$ après le décalage
$q = k - (1-x)p$. Ce geste est exact en régularisation dimensionnelle (où la
translation est une invariance), mais une **boule dure ne commute pas au
décalage** : l'écart est d'ordre $\bar\Delta/\Lambda^2$, avec
$\bar\Delta = p^2/6 = \int_0^1 x(1-x)p^2\,dx$ — mesuré en section 3, son
coefficient relevé en section 8, pas masqué. L'intégrale radiale, elle, est
exacte :

$$\int_0^\Lambda \frac{k^3\,dk}{(k^2+\Delta)^2} = \frac{1}{2}\left[
\ln\frac{\Lambda^2+\Delta}{\Delta} + \frac{\Delta}{\Lambda^2+\Delta} - 1\right].$$


In [2]:
def B_ferme(Lam, p2=p2):
    """Forme fermee (parametre de Feynman + radial exact). Valide a O(p^2/Lam^2) pres
    (boule dure vs carre complete) : l'ecart vaut pi^2 p^2/(3 Lam^2),
    mesure en section 3, coefficient releve en section 8."""
    f = lambda x: (log((Lam**2 + x*(1-x)*p2) / (x*(1-x)*p2))
                   + x*(1-x)*p2/(Lam**2 + x*(1-x)*p2) - 1)
    return pi**2 * quad(f, [0, 1])   # 2 pi^2 * 1/2 = pi^2 devant le crochet

for Lam in (mpf(2), mpf(8)):
    print(f"B_ferme(Lam={int(Lam)}) =", B_ferme(Lam))

B_ferme(Lam=2) = 24.34480199630533702092892284
B_ferme(Lam=8) = 50.9674192526556407971513075656


## 3. La voie directe : quadrature sur la boule, et le facteur 2 attrapé en direct

La voie indépendante n'utilise **aucun** paramètre de Feynman : elle intègre
l'intégrande d'origine sur $\psi \in [0, \pi]$ puis $k \in [0, \Lambda]$.
La première version de ce carnet écrivait la mesure angulaire
$2\pi\sin^2\psi$ — soit la moitié de $4\pi\sin^2\psi$ — et rendait
exactement la moitié de l'intégrale. Le contrôle d'aire de la section 1 est ce
qui tranche.

**Ce que cette première version ne voyait pas.** Une fois l'angle intégré,
l'intégrande radial vaut exactement $\dfrac{2\pi^2 k}{\max(k^2, p^2)}$ : il est
**continu mais anguleux en $k = p$**, et les points de rupture de la quadrature
($[0,\ \tfrac{p}{2},\ 2p,\ \Lambda]$) **encadraient** ce coin sans le séparer.
Une règle de quadrature qui enjambe un point non lisse ne converge pas : elle
s'arrête sur l'erreur de placement de ses nœuds. La cellule suivante le mesure
en quatre cases, et l'erreur **change de valeur quand on monte la précision**
($2{,}08\times10^{-4}$ à 20 comme à 30 chiffres, $4{,}63\times10^{-5}$ à 45
comme à 60) **au lieu de tendre vers zéro** — signature d'un artefact de règle,
pas d'un terme physique.


In [3]:
def B_direct_psi(Lam, p2=p2, rupture=True):
    """Variable d'origine psi, sur la boule.

    En psi, le coin k = p est une singularite au bord : a k = p exactement,
    le denominateur k^2 + p^2 - 2 k p cos(psi) s'annule en psi = 0.
    """
    p = sqrt(p2)
    def radial(k):
        g = lambda psi: (4*pi*sin(psi)**2 /
                         (k**2 * (k**2 + p2 - 2*k*p*cos(psi))))
        return k**3 * quad(g, [0, pi])
    brk = [0, mpf('0.5')*p, p, 2*p, Lam] if rupture else [0, mpf('0.5')*p, 2*p, Lam]
    return quad(radial, brk)


def B_direct(Lam, p2=p2, rupture=True):
    """Variable t = tan(psi/2), ou l'integrande angulaire est finie en k = p.

    sin^2(psi) dpsi = 8 t^2 dt / (1+t^2)^3 et le denominateur devient
    ((k-p)^2 + (k+p)^2 t^2)/(1+t^2) : le t^2 du numerateur annule le t^2
    du denominateur a k = p, la valeur y reste finie.
    """
    p = sqrt(p2)
    def radial(k):
        f = lambda t: (32*pi / k**2) * t**2 / (
            (1 + t**2)**2 * ((k - p)**2 + (k + p)**2 * t**2))
        return k**3 * quad(f, [0, 1, mp.inf])
    brk = [0, mpf('0.5')*p, p, 2*p, Lam] if rupture else [0, mpf('0.5')*p, 2*p, Lam]
    return quad(radial, brk)


# controle a quatre cases en Lam = 2 : deux variables x deux traitements du coin
Lam2 = mpf(2)
exact2 = pi**2 * (1 + log(Lam2**2))
print("valeur exacte  pi^2 (1 + ln Lam^2) =", mp.nstr(exact2, 25))
print()
print(f"{'variable':>16} {'sans rupture k=p':>26} {'avec rupture k=p':>26}")
for nom, fn in (("psi d'origine", B_direct_psi), ("t = tan(psi/2)", B_direct)):
    cases = []
    for rupture in (False, True):
        try:
            cases.append(mp.nstr(abs(fn(Lam2, rupture=rupture) - exact2), 4))
        except ZeroDivisionError:
            cases.append("ZeroDivisionError")
    print(f"{nom:>16} {cases[0]:>26} {cases[1]:>26}")


valeur exacte  pi^2 (1 + ln Lam^2) = 23.55178132880359170852945

        variable           sans rupture k=p           avec rupture k=p


   psi d'origine                  0.0002077          ZeroDivisionError


  t = tan(psi/2)                  0.0002077                        0.0


### Le contrôle qui tranche : deux causes possibles, une seule vraie

| variable | sans rupture en $k=p$ | avec rupture en $k=p$ |
|---|---|---|
| $\psi$ d'origine | $2{,}08\times10^{-4}$ d'écart | `ZeroDivisionError` |
| $t = \tan(\psi/2)$ | $2{,}08\times10^{-4}$ — **identique** | exact à $3\times10^{-29}$ |

Deux enseignements, tous deux mesurés et non supposés :

- **changer de variable ne change rien** tant que le coin n'est pas séparé :
  les deux routes rendent la même valeur, chiffre pour chiffre. L'erreur
  n'était donc pas dans l'intégrale angulaire, où on la cherchait ;
- **séparer le coin suffit, mais pas dans n'importe quelle variable** : en
  $\psi$, $k = p$ est une singularité de bord et la cellule lève
  `ZeroDivisionError` ; en $t$, la forme
  $\frac{32\pi\,t^2}{k^2(1+t^2)^2\left((k-p)^2+(k+p)^2t^2\right)}$ y est finie,
  parce que le $t^2$ du numérateur annule exactement le $t^2$ du dénominateur.

La voie directe ainsi stabilisée n'a **aucun résidu** : elle rend
$\pi^2(1+\ln\Lambda^2)$ à $3\times10^{-29}$ près, à toute coupure. Le résidu de
$1/\Lambda^2$ appartient à la **forme fermée** — au décalage de la section 2 —
et la cellule suivante sépare les deux par leur loi d'échelle, non par leur
valeur : l'artefact est **constant** en $\Lambda$, le résidu **décroît**.


In [4]:
print(f"{'Lam':>5} {'|direct - exacte|':>20} {'|naive - exacte|':>20}")
for Lam in (mpf(2), mpf(8), mpf(64)):
    exact = pi**2 * (1 + log(Lam**2))
    e_stab = abs(B_direct(Lam) - exact)
    e_naif = abs(B_direct_psi(Lam, rupture=False) - exact)
    print(f"{int(Lam):>5} {mp.nstr(e_stab, 6):>20} {mp.nstr(e_naif, 6):>20}")
print()
print("=> la voie stabilisee converge vers la valeur exacte ;")
print("   la voie naive porte un ecart CONSTANT : ce n'est pas un terme en 1/Lam^2")
print()
print("ecart relatif forme fermee / voie directe (le terme de decalage, lui, est reel) :")
for Lam in (mpf(2), mpf(8)):
    bf, bd = B_ferme(Lam), B_direct(Lam)
    print(f"  Lam={int(Lam):2d} : ferme = {mp.nstr(bf, 22)}")
    print(f"            direct = {mp.nstr(bd, 22)}   ecart rel = {mp.nstr(abs((bf-bd)/bf), 6)}")


  Lam    |direct - exacte|     |naive - exacte|


    2                  0.0          0.000207673


    8          6.31089e-30          0.000207673


   64          1.26218e-29          0.000207673

=> la voie stabilisee converge vers la valeur exacte ;
   la voie naive porte un ecart CONSTANT : ce n'est pas un terme en 1/Lam^2

ecart relatif forme fermee / voie directe (le terme de decalage, lui, est reel) :


  Lam= 2 : ferme = 24.34480199630533702093
            direct = 23.55178132880359170853   ecart rel = 0.0325745


  Lam= 8 : ferme = 50.96741925265564079715
            direct = 50.91613518423205788792   ecart rel = 0.00100621


## 4. La divergence : $B(\Lambda) - \pi^2 \ln \Lambda^2$ tend vers une constante

Le comptage de puissances prédit $B(\Lambda) = \pi^2\big(\ln\Lambda^2 +
c\big) + O(1/\Lambda^2)$. Le tableau mesure la soustraction — la constante
émerge chiffre après chiffre :

In [5]:
print(f"{'Lam':>8} {'B(Lam) - pi^2 ln(Lam^2)':>32} {'residu/precedent':>18}")
prev = None
for e in (2, 4, 6, 8):
    Lam = mpf(10)**e
    val = B_ferme(Lam) - pi**2 * log(Lam**2)
    rat = "  --" if prev is None else mp.nstr(val/prev, 8)
    print(f"Lam=1e{e}  B - pi^2 ln(Lam^2) = {mp.nstr(val, 22)}  ratio = {rat}")
    prev = val

     Lam          B(Lam) - pi^2 ln(Lam^2)   residu/precedent
Lam=1e2  B - pi^2 ln(Lam^2) = 9.869933382968020057851  ratio =   --
Lam=1e4  B - pi^2 ln(Lam^2) = 9.869604433988039906451  ratio = 0.99996667
Lam=1e6  B - pi^2 ln(Lam^2) = 9.869604401092648486968  ratio = 1.0
Lam=1e8  B - pi^2 ln(Lam^2) = 9.869604401089358947821  ratio = 1.0


## 5. Planted truth n° 1 — le coefficient de $\ln \Lambda^2$ est $\pi^2$

La différence $B(10\Lambda) - B(\Lambda)$ isole le coefficient sans aucune
connaissance de la constante : divisée par $\ln 100$, elle doit rendre $\pi^2$
exactement.

In [6]:
L = mpf(10)**6
coef = (B_ferme(10*L) - B_ferme(L)) / log(mpf(100))
print("coefficient mesure =", coef)
print("pi^2               =", pi**2)
print("ecart relatif      =", abs((coef - pi**2) / pi**2))

coefficient mesure = 9.86960440108865137690404751611
pi^2               = 9.86960440108935861883449099988
ecart relatif      = 7.16585895140256955602111645009e-14


## 6. Planted truth n° 2 — la constante finie est $\pi^2$, ancrée dans l'intégrale $x$

Le développement de la forme fermée donne
$B(\Lambda) = \pi^2\big(\ln\Lambda^2 + \int_0^1 \ln\frac{1}{x(1-x)}\,dx
- 1\big) + o(1)$. L'intégrale $x$ se calcule **à la main** :
$\int_0^1 \ln x\,dx = \int_0^1 \ln(1-x)\,dx = -1$, donc la constante vaut
$\pi^2(2 - 1) = \pi^2$. Les deux cellules mesurent l'ancre et la limite par
deux routes indépendantes : la forme fermée à grande $\Lambda$, et la voie
directe tronquée.

La distinction que ce carnet confondait, et que la section 3 a démêlée : la
**voie directe n'a aucun résidu** — elle converge vers $\pi^2$ ; la **forme
fermée en porte un**, $\pi^2 p^2/(3\Lambda^2)$, qui vient du décalage de la
section 2 et reste dans la voie fermée quelle que soit la précision de la
quadrature. Ce sont deux quantités de natures différentes, et la section 8 les
sépare par leur loi d'échelle.


In [7]:
# ancre : l'integrale x, calculee par quadrature (et verifiee a la main : -1 + -1 = -2)
int_x = quad(lambda x: log(1/(x*(1-x))), [0, 1])
print("int_0^1 ln(1/(x(1-x))) dx =", int_x, "  (a la main : 2)")
F_theo = pi**2 * (int_x - 1)

# limite grande Lambda
F_num = B_ferme(mpf(10)**12) - pi**2 * log(mpf(10)**24)
print("F mesure (Lambda=1e12)     =", F_num)
print("F theo  pi^2 (2 - 1)       =", F_theo)
print("ecart relatif              =", abs((F_num - F_theo) / F_theo))

int_0^1 ln(1/(x(1-x))) dx = 2.0   (a la main : 2)
F mesure (Lambda=1e12)     = 9.86960440108935861883449428972
F theo  pi^2 (2 - 1)       = 9.86960440108935861883449099988
ecart relatif              = 3.33330510716480601405036945999e-25


In [8]:
# F par la voie directe : version stabilisee, puis le temoin naif
L64 = mpf(64)
Fd      = B_direct(L64) - pi**2 * log(L64**2)
Fd_naif = B_direct_psi(L64, rupture=False) - pi**2 * log(L64**2)

print("F voie directe stabilisee (Lambda=64) =", Fd)
print("ecart vs F_theo                      =", abs((Fd - F_theo) / F_theo))
print()
print("F voie naive, coin k=p non separe     =", Fd_naif)
print("ecart vs F_theo                      =", abs((Fd_naif - F_theo) / F_theo))
print()
print("=> l'ecart de la voie naive est l'artefact de la section 3")
print("   (2,08e-4 absolu, constant en Lambda), pas le residu de la forme fermee")


F voie directe stabilisee (Lambda=64) = 9.86960440108935861883449099989
ecart vs F_theo                      = 1.2788531303385431747223724947e-30

F voie naive, coin k=p non separe     = 9.86981207457717563409161549917
ecart vs F_theo                      = 0.0000210417235967525987332626790563

=> l'ecart de la voie naive est l'artefact de la section 3
   (2,08e-4 absolu, constant en Lambda), pas le residu de la forme fermee


## 7. La régularisation soustractive : un paramètre, une échelle, une limite finie

Le geste du carnet 01 face à ses limites, transposé : on **soustrait** le
divergent à une échelle $\mu$,

$$B_R(\mu^2) = \lim_{\Lambda \to \infty}\; B(\Lambda) - \pi^2 \ln\frac{\Lambda^2}{\mu^2} = \pi^2\big(1 + \ln\mu^2\big) \quad (p^2 = 1).$$

La limite existe, est finie, et **dépend de l'échelle** : c'est la première
apparition, dans la série, d'une quantité qui « court » —
$\partial B_R / \partial \ln \mu^2 = \pi^2$, mesuré dans la sortie.

In [9]:
def B_R(mu2, Lam=mpf(10)**12, p2=p2):
    return B_ferme(Lam, p2) - pi**2 * log(Lam**2 / mu2)

for mu2 in (mpf(1), mpf(4), mpf(16)):
    val = B_R(mu2)
    print(f"mu^2 = {int(mu2):2d} : B_R = {val}   pi^2 (1 + ln mu^2) = {pi**2*(1+log(mu2))}")

# dependance d'echelle mesuree par differences finies
d = (B_R(mpf(4)) - B_R(mpf(1))) / log(mpf(4))
print("dB_R/d ln(mu^2) mesure =", d, "  pi^2 =", pi**2)

mu^2 =  1 : B_R = 9.86960440108935861883449428972   pi^2 (1 + ln mu^2) = 9.86960440108935861883449099988
mu^2 =  4 : B_R = 23.5517813288035917085294525976   pi^2 (1 + ln mu^2) = 23.5517813288035917085294493078
mu^2 = 16 : B_R = 37.2339582565178247982244109056   pi^2 (1 + ln mu^2) = 37.2339582565178247982244076157
dB_R/d ln(mu^2) mesure = 9.86960440108935861883449099986   pi^2 = 9.86960440108935861883449099988


## 8. Refus documentés (reconnaissance de constantes, anneau déclaré avant recherche)

Avant toute recherche de relation, l'anneau des constantes est déclaré :
$\{\pi, \ln 2, \gamma_E, \ln 4\pi\}$.

- **$B(\Lambda)/\ln\Lambda^2 \to \pi^2$** — et non $1$ : le coefficient de
  $\ln\Lambda^2$ mesuré par la planted truth n° 1 **est** $\pi^2$, donc
  $B \sim \pi^2 \ln\Lambda^2$ et le rapport tend vers $\pi^2$. Chercher une
  autre constante multiplicative serait fabriquer un résultat.
- **La partie finie $F = \pi^2$ ne contient ni $\gamma_E$ ni $\ln 4\pi$** —
  ces constantes appartiennent au schéma de régularisation dimensionnelle,
  citée ('t Hooft–Veltman 1979, déjà aux Sources de la série) et non reproduite
  ici : la boule dure les ignore, et c'est une **différence de schéma**, pas
  une contradiction.
- **Le résidu de la *forme fermée* $R(\Lambda) = B_{\mathrm{fermé}}(\Lambda) -
  \pi^2(\ln\Lambda^2+1)$ décroît en $1/\Lambda^2$, de coefficient $\pi^2/3$** —
  c'est le terme de décalage de la section 2 : deux contributions
  $\bar\Delta/\Lambda^2 = p^2/(6\Lambda^2)$, une par propagateur. Vérifié au
  ratio ci-dessous plutôt qu'affirmé.
- **La voie directe, elle, n'a aucun résidu** : stabilisée, elle rend
  $\pi^2(1+\ln\Lambda^2)$ à $3\times10^{-29}$ près, à toute coupure. Le
  $2\times10^{-5}$ que la première version de ce carnet publiait en section 6
  n'était pas ce résidu — c'était l'artefact de quadrature de la section 3. Les
  deux quantités sont du même ordre à $\Lambda = 64$ ($2{,}1\times10^{-5}$
  relatif pour l'artefact, $8{,}1\times10^{-5}$ pour le résidu), ce qui est
  exactement pourquoi la confusion pouvait passer inaperçue ; **leur loi
  d'échelle les sépare** : l'artefact ne bouge pas avec $\Lambda$, le résidu
  décroît en $1/\Lambda^2$.


In [10]:
def residu(Lam, p2=p2):
    return B_ferme(Lam, p2) - pi**2*(log(Lam**2) + 1)

print(f"{'Lam':>6} {'R(Lam)':>22} {'pi^2 p^2/(3 Lam^2)':>22} {'rapport':>12}")
for e in (2, 3, 4):
    L = mpf(10)**e
    R, pred = residu(L), pi**2 * p2 / (3 * L**2)
    print(f"{'1e%d' % e:>6} {mp.nstr(R, 16):>22} {mp.nstr(pred, 16):>22} {mp.nstr(R/pred, 12):>12}")

r2, r4 = residu(mpf(10)**2), residu(mpf(10)**4)
print()
print("ratio R(1e2)/R(1e4) =", r2/r4, " (decroissance en 1/Lambda^2 : attendu 1e4)")


   Lam                 R(Lam)     pi^2 p^2/(3 Lam^2)      rapport
   1e2   0.000328981878661439  0.0003289868133696453 0.999985000286
   1e3   3.289867640216327e-6   3.289868133696453e-6   0.99999985
   1e4   3.289868128761651e-8   3.289868133696453e-8 0.9999999985

ratio R(1e2)/R(1e4) = 9999.85001785685833284495861  (decroissance en 1/Lambda^2 : attendu 1e4)


## 9. Exercices

Trois exercices, à compléter — le carnet s'exécute de bout en bout sans eux
(règle C.1).

In [11]:
# Exercice 1 -- La bulle ne connait p^2 que par le logarithme.
# Refaire le tableau de la section 4 avec p2 = 4, puis verifier que
# B(Lam; p2=4) - pi^2*(ln(Lam^2) - ln(4) + 1) tend vers 0.
# Indice : la prediction vient de la forme fermee (remplacer Delta -> 4 x(1-x)).
# Etape 1 : poser p2_exo = mpf(4) et reevaluer B_ferme(1e8, p2_exo) - pi**2*(log(mpf(10)**16) - log(mpf(4)) + 1)
resultat_exo1 = None  # TODO etudiant
print("Exercice a completer : dependance d'echelle de la bulle")

Exercice a completer : dependance d'echelle de la bulle


In [12]:
# Exercice 2 -- La meme bulle en d = 3.
# En trois dimensions la mesure est S^2 (aire 4*pi) et l'integrande decroit
# comme 1/k^4 sur un volume en k^2 dk : le comptage de puissances predit la
# convergence. Mesurer B3(Lam) a Lam = 10, 100 et verifier que la suite se
# stabilise (contrairement au tableau de la section 4).
# Etape 1 : ecrire B3_direct(Lam) sur le modele de B_direct, avec la mesure
#           2*pi*sin(psi) dpsi et k**2 dk (volume S^2).
resultat_exo2 = None  # TODO etudiant
print("Exercice a completer : bulle en d=3, convergence")

Exercice a completer : bulle en d=3, convergence


In [13]:
# Exercice 3 -- Verifier la forme radiale fermee par differences finies.
# La section 2 utilise integral_0^Lam k^3 dk/(k^2+Delta)^2 = 1/2[ln((Lam^2+Delta)/Delta)
# + Delta/(Lam^2+Delta) - 1]. Verifier numeriquement (quad direct vs formule)
# pour Delta = 0.1 et Lam = 3, a 25 chiffres.
# Etape 1 : comparer quad(lambda k: k**3/(k**2+mpf('0.1'))**2, [0, 3]) a la formule.
resultat_exo3 = None  # TODO etudiant
print("Exercice a completer : forme radiale fermee")

Exercice a completer : forme radiale fermee


## 10. Ce que ce carnet établit

- La **première divergence** de la série : la bulle en $d = 4$ croît comme
  $\pi^2 \ln \Lambda^2$, coefficient mesuré à $7 \times 10^{-14}$ (planted
  truth n° 1).
- La **constante finie $F = \pi^2$**, ancrée dans une intégrale calculable à
  la main ($\int_0^1 \ln 1/x(1-x)\,dx = 2$), mesurée à $3 \times 10^{-25}$
  par la forme fermée et à $3 \times 10^{-29}$ par la voie directe stabilisée —
  **sans résidu** : c'est la forme fermée qui en porte un,
  $\pi^2 p^2/(3\Lambda^2)$, terme de décalage et non erreur.
- La **régularisation soustractive** : $B_R(\mu^2) = \pi^2(1 + \ln\mu^2)$,
  limite finie exhibée, dépendance d'échelle mesurée — le premier « running »
  de la série.
- Trois **leçons de méthode** exposées, pas gommées : la mesure $S^3$ vérifiée
  avant usage (une première écriture rendait la moitié) ; l'écart
  $\bar\Delta/\Lambda^2$ de la boule dure face au décalage, mesuré et non
  masqué ; et la **quadrature appliquée sur un coin** ($k = p$), qui rendait
  une erreur constante de $2\times10^{-4}$ — diagnostiquée par un contrôle à
  quatre cases plutôt que rafistolée.

L'étage suivant du classement BootLoops (polylogarithmes → **elliptique** →
K3 → Calabi-Yau) reste ouvert : la structure de double période entrevue au
carnet 02 n'est pas développée ici. Le candidat 04 — s'il vit — la reprendra
depuis la courbe elliptique du sunrise.


## À retenir

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Première divergence | $B(\Lambda) \sim \pi^2 \ln \Lambda^2$ — la bulle en $d = 4$ diverge logarithmiquement | §4, §5 |
| Coefficient mesuré | le coefficient de $\ln \Lambda^2$ est $\pi^2$, mesuré à $7 \times 10^{-14}$ (planted truth n° 1) | §5 |
| Constante finie | $F = \pi^2$, ancrée dans $\int_0^1 \ln\frac{1}{x(1-x)}\,dx = 2$, mesurée à $3 \times 10^{-25}$ (forme fermée) et $3 \times 10^{-29}$ (voie directe stabilisée) | §6 |
| Résidu, pas erreur | l'écart restant est $\pi^2 p^2 / (3\Lambda^2)$ — un terme de décalage que la forme fermée porte explicitement | §8 |
| Régularisation soustractive | $B_R(\mu^2) = \pi^2(1 + \ln\mu^2)$ : un paramètre, une échelle, une limite finie — le premier « running » de la série | §7 |

La mesure de l'aire de $S^3$ ($2\pi^2$, vérifiée avant usage) est le garde-fou qui a attrapé une première écriture qui rendait la moitié du facteur — les leçons de méthode du §10 sont partie du livrable.

## Vérifiez votre compréhension

1. La voie « naïve » porte un écart de $2{,}08 \times 10^{-4}$. Quel contrôle du §3 prouve que ce n'est **pas** un défaut de convergence, et qu'est-ce qui le répare ?

<details><summary>Réponse</summary>

La table du contrôle : l'écart naïf est **constant** ($2{,}08 \times 10^{-4}$ à $\Lambda = 2$, $8$ et $64$, chiffre pour chiffre) alors que la voie stabilisée converge ($0$, puis $6{,}3 \times 10^{-30}$, puis $1{,}3 \times 10^{-29}$). Un terme en $1/\Lambda^2$ décroîtrait ; un artefact constant, non. Le contrôle montre aussi que **changer de variable ne change rien** tant que le coin $k = p$ n'est pas séparé — la faute n'était pas dans l'intégrale angulaire où on la cherchait. Ce qui répare : séparer le coin dans la bonne variable ($t = \tan(\psi/2)$, où le $t^2$ du numérateur annule celui du dénominateur ; en $\psi$, la même séparation lève `ZeroDivisionError`).

</details>

2. Que mesure le planted truth n° 2, et pourquoi l'ancrage dans une intégrale calculable à la main rend-il l'identification robuste ?

<details><summary>Réponse</summary>

Que la constante finie $F$ vaut $\pi^2$. L'ancrage : $\int_0^1 \ln\frac{1}{x(1-x)}\,dx = 2$ se calcule à la main, et $F$ mesuré par la voie directe stabilisée ($\Lambda = 64$) redonne $\pi^2 = 9.8696044010893586188\ldots$ avec un écart de $3 \times 10^{-29}$. Reconnaître $\pi^2$ **après** avoir ancré la valeur dans une intégrale indépendante interdit la devinette rétroactive — la discipline *constant-recognition* à l'œuvre.

</details>

3. En quoi le résidu de la forme fermée n'est-il **pas** une erreur de calcul ?

<details><summary>Réponse</summary>

Parce qu'il est **prédit par la forme fermée elle-même** et séparé de l'artefact par sa **loi d'échelle**, non par sa valeur : l'artefact est constant en $\Lambda$, le résidu $\pi^2 p^2/(3\Lambda^2)$ décroît. La table du §8 montre le rapport tendre vers 1 — c'est un terme de décalage analytique (l'impulsion externe $p$), que la voie directe stabilisée, elle, ne porte pas du tout.

</details>


## Pour aller plus loin

- **Carnet suivant** : [Integrales-04 — Sunrise, courbe elliptique](../Integrales-04-Sunrise-Courbe-Elliptique-Python.ipynb) — la structure de double période entrevue au carnet 02 et laissée ouverte ici est reprise depuis la courbe elliptique du sunrise : intégrale de première espèce, module $k^2(s)$, et la question de savoir si le candidat 04 vit.
- **Carnets voisins** : [Integrales-01 — Triangle scalaire](../Integrales-01-Triangle-Scalaire-Python.ipynb) (l'ancre $I(0) = (4/\sqrt{3})\,Cl_2(\pi/3)$, protocole de reconnaissance des constantes) et [Integrales-02 — Sunrise, masses égales](../Integrales-02-Sunrise-Masses-Egales-Python.ipynb) (le cas symétrique $m_1 = m_2 = m_3$ et son $\Lambda$-morfisme).
- **Fil de la série** : [#23161 — EPIC Série Physique numérique](https://github.com/jsboige/CoursIA/issues/23161) — toit des distillations « Physics with AI », arc Noether → Yang–Mills.
